# Named Graph / Quad Support — End-to-End Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neo4j-labs/rdflib-neo4j/blob/main/notebooks/named_graphs_demo.ipynb)

This notebook demonstrates `named_graphs=True` in `Neo4jStoreConfig` — opt-in support
for RDF named graphs (quads) that stores each triple alongside the graph URI it came from.

### The story

A research organisation publishes data in two separate named graphs:
- **`http://org/graph/people`** — employee directory (persons, departments)
- **`http://org/graph/projects`** — project assignments (who works on what)

We load both graphs into Neo4j, then query them independently and together.

### What named graphs do in Neo4j

Each node and relationship gets a `graphUri` property recording which named graph it
came from. The same URI (e.g. `ex:alice`) written from two different graphs produces
**two separate Neo4j nodes** — one per graph — allowing per-graph isolation and queries.

> ⚠️ **Switching modes**: changing `named_graphs=True/False` on an existing database
> requires a full re-import. Nodes written without `graphUri` won't be queryable by graph.

## 0. Setup

In [ ]:
%pip install -q rdflib-neo4j neo4j python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()  # reads NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE from .env

URI      = os.environ["NEO4J_URI"]
USER     = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f"Connected to {URI} / {DATABASE}")

## 1. The Dataset — Two Named Graphs

We use **TriG** format (Turtle extended with named graph blocks).

In [ ]:
TRIG = """
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex:   <http://example.org/> .
@prefix org:  <http://org/> .

# Graph 1: employee directory
<http://org/graph/people> {
    ex:alice a foaf:Person ;
        foaf:name "Alice" ;
        foaf:age  30 ;
        org:department "engineering" .

    ex:bob a foaf:Person ;
        foaf:name "Bob" ;
        foaf:age  25 ;
        org:department "product" .

    ex:carol a foaf:Person ;
        foaf:name "Carol" ;
        foaf:age  35 ;
        org:department "engineering" .
}

# Graph 2: project assignments
<http://org/graph/projects> {
    ex:project_a a org:Project ;
        org:name   "Graph Analytics Platform" ;
        org:status "active" .

    ex:project_b a org:Project ;
        org:name   "Data Ingestion Pipeline" ;
        org:status "active" .

    ex:alice org:worksOn ex:project_a .
    ex:carol org:worksOn ex:project_a .
    ex:bob   org:worksOn ex:project_b .
}
"""

## 2. Load with Named Graph Support

Use `ConjunctiveGraph` (quad-aware) and set `named_graphs=True` in the config.
Each named graph block in TriG becomes a separate RDF context.

In [ ]:
from rdflib import ConjunctiveGraph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

# Clear previous data + ensure index (named_graphs uses a plain index, not uniqueness)
driver.execute_query("MATCH (n:Resource) DETACH DELETE n", database_=DATABASE)
driver.execute_query(
    "CREATE INDEX resource_uri IF NOT EXISTS FOR (r:Resource) ON (r.uri)",
    database_=DATABASE,
)

auth_data = {"uri": URI, "database": DATABASE, "user": USER, "pwd": PASSWORD}
config = Neo4jStoreConfig(
    auth_data=auth_data,
    custom_prefixes={
        "foaf": "http://xmlns.com/foaf/0.1/",
        "org":  "http://org/",
    },
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    named_graphs=True,   # ← enable quad support
    batching=False,
)

cg = ConjunctiveGraph(store=Neo4jStore(config=config))
cg.open(config, create=True)
cg.parse(data=TRIG, format="trig")
cg.close(True)

# Verify — all Resource nodes with a graphUri property
records, _, _ = driver.execute_query(
    "MATCH (n:Resource) RETURN n.uri AS uri, n.graphUri AS graph ORDER BY graph, uri",
    database_=DATABASE,
)
print(f"{'URI':45}  GRAPH")
print("-" * 80)
for r in records:
    print(f"{r['uri']:45}  {r['graph']}")

## 3. Same URI in Two Graphs = Two Nodes

`ex:alice` appears in BOTH graphs (as a `Person` in people graph, and as a project
assignee in projects graph). With `named_graphs=True` these are **two separate nodes**.

In [ ]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Resource {uri: 'http://example.org/alice'})
    RETURN n.graphUri AS graph, labels(n) AS labels, keys(n) AS props
    """,
    database_=DATABASE,
)
print(f"Found {len(records)} node(s) for ex:alice:")
for r in records:
    print(f"  graph={r['graph']}")
    print(f"  labels={r['labels']}")
    print(f"  properties={r['props']}")
    print()

## 4. Query a Single Named Graph

Filter by `graphUri` to scope your query to one graph.

In [ ]:
# People graph only
records, _, _ = driver.execute_query(
    """
    MATCH (p:Person {graphUri: 'http://org/graph/people'})
    RETURN p.name AS name, p.age AS age, p.department AS dept
    ORDER BY p.name
    """,
    database_=DATABASE,
)
print("=== People graph ===")
for r in records:
    print(f"  {r['name']:8}  age={r['age']}  dept={r['dept']}")

In [ ]:
# Projects graph only
records, _, _ = driver.execute_query(
    """
    MATCH (proj:Project {graphUri: 'http://org/graph/projects'})
    RETURN proj.name AS project, proj.status AS status
    ORDER BY project
    """,
    database_=DATABASE,
)
print("=== Projects graph ===")
for r in records:
    print(f"  {r['project']:35}  [{r['status']}]")

## 5. Cross-Graph Query — Join People and Projects

Even though the two graphs produce separate nodes for the same URIs, we can
join across them by matching on the `uri` property.

In [ ]:
records, _, _ = driver.execute_query(
    """
    // Person from the people graph
    MATCH (person:Person {graphUri: 'http://org/graph/people'})
    // Same URI in the projects graph
    MATCH (assignee:Resource {uri: person.uri, graphUri: 'http://org/graph/projects'})
    // Their project assignment
    MATCH (assignee)-[:worksOn]->(proj:Project {graphUri: 'http://org/graph/projects'})
    RETURN person.name AS name, person.department AS dept, proj.name AS project
    ORDER BY name
    """,
    database_=DATABASE,
)
print(f"{'Name':8}  {'Dept':12}  Project")
print("-" * 60)
for r in records:
    print(f"{r['name']:8}  {r['dept']:12}  {r['project']}")

## 6. Contrast with Default Mode (no named graphs)

For comparison, import the same data *without* `named_graphs=True`.
Now `ex:alice` maps to a **single node** — no `graphUri`, last-write wins.

In [ ]:
# Side-by-side comparison (we DON'T commit this — just preview)
from rdflib_neo4j import Neo4jStoreConfig

config_no_ng = Neo4jStoreConfig(
    custom_prefixes={
        "foaf": "http://xmlns.com/foaf/0.1/",
        "org":  "http://org/",
    },
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    named_graphs=False,  # default
    preview=True,        # dry-run: capture queries without executing
)

from rdflib import Graph
from rdflib_neo4j import Neo4jStore

g_preview = Graph(store=Neo4jStore(config=config_no_ng))
g_preview.parse(data=TRIG, format="trig")

print("Without named_graphs — first 3 queries:")
for q, p in config_no_ng.get_preview_results()[:3]:
    print("  ", q[:80], "..." if len(q) > 80 else "")
    if "graphUri" in str(p):
        print("    ← has graphUri")
    else:
        print("    ← NO graphUri")

## 7. rdflib-level Named Graph Access

You can also use rdflib's `ConjunctiveGraph.contexts()` and context-scoped queries
to access named graphs through the rdflib API.

In [ ]:
from rdflib import URIRef, ConjunctiveGraph
from rdflib_neo4j import Neo4jStore

# Re-open in read mode
cg2 = ConjunctiveGraph(store=Neo4jStore(config=config))
cg2.open(config, create=False)

people_graph = URIRef("http://org/graph/people")
people_ctx = cg2.get_context(people_graph)

from rdflib import RDF
foaf_Person = URIRef("http://xmlns.com/foaf/0.1/Person")

print("Persons in the people graph (via rdflib API):")
for s, p, o in people_ctx.triples((None, RDF.type, foaf_Person)):
    print(" ", s)

cg2.close()

In [ ]:
driver.close()